# 📈 Proyecto final Semillero: Mesa de ayuda IA para el Departamento de Ventas de Patito S.A. - Agentes con LangChain + Gemini

## *Grupo Modo Avión ✈️*
### *Integrantes*
- ALVARADO MARÍA GISSELIE
- HUANCA ASHLEY BRIGGITTE
- YUGSAN LEONARDO MATEO

Nuestro equipo ha sido contratado por **Patito S.A.** para optimizar e implementar inteligencia artificial en el departamento de Ventas.

El encargo consiste en desarrollar un prototipo de mesa de ayuda IA para Ventas, compuesto por agentes especializados (LangChain) que colaboren entre sí para responder preguntas comerciales usando una base documental ficticia. Hoy construimos un asistente con LangChain y Google Gemini compuesto por cinco agentes coordinados por un orquestador:
| # | Agente | Que hace | Tecnologia |
|---|---|---|---|
| 1 | **Catalogo y precios** | Respondes consultas sobre productos y precios. | Embeddings Gemini + Chroma |
| 2 | **Politicas comerciales**| Responde consultas sobre descuentos, niveles de autorización. | Embeddings Gemini + Chroma |
| 3 | **Procesas de ventas y CRM**| Responde consultas sobre etapas del proceso comercial, registro de oportunidades. | Embeddings Gemini + Chroma |
| 4 | **Multimodal de imagen** | Lee un recibo/factura y extrae sus datos | Gemini con vision. |
| 5 | **Accion (registro)** | Registra la solicitud en un `.txt` con validacion. | `@tool` + function calling |
| ⭐ | **Orquestador** | Decide que agente usa en cada consulta. | `create_agent` (LangChain 1.x) |


Nuestro objetivo no es construir una solución productiva completa, sino evidenciar los conocimientos adquiridos en el semillero: arquitectura, manejo de RAG con agentes LangChain, uso de Google Gemini como LLM y embeddings, calidad de código y capacidad de explicación.


## 1. Instalacción de dependencias obligatorias

In [27]:
# LangChain + Google Gemini + ChromaDB
!pip install -q langchain langchain-google-genai langchain-community langchain-chroma chromadb langchain-text-splitters
# Utilidades
!pip install -q pillow pandas ipywidgets python-dotenv
# Observabilidad con Phoenix
!pip install -q arize-phoenix openinference-instrumentation-langchain
print("Instalación completa.")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Instalación completa.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import sys

!{sys.executable} -m pip install mypy_extensions


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Conexion a Google Gemini

Configuramos **un mismo proveedor (Gemini)** para todo:

- **LLM:** `gemini-2.0-flash` — es **multimodal**, asi que sirve tanto para texto como para leer imagenes.
- **Embeddings:** `models/text-embedding-004` — convierte el texto de la politica en vectores.

### 🔑 API Key
1. Ve a [Google AI Studio](https://aistudio.google.com/apikey)
2. Inicia sesion y haz clic en **Create API Key**
3. Copia la clave y pegala abajo (o usa `getpass` para no dejarla escrita).

In [29]:
import os, getpass
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Opcion A: pegar la clave directamente (reemplaza el texto)
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6LMhfK8L85Y6OS_pFMzoBepk1ddnYitpe18YvkaSEzb1g"
# Opcion B (mas segura): descomenta la siguiente linea y comenta la de arriba
#os.environ["GOOGLE_API_KEY"] = getpass.getpass("")

MODELO_LLM = "gemini-3.1-flash-lite"
MODELO_EMBEDDING = "gemini-embedding-2-preview"

llm = ChatGoogleGenerativeAI(model=MODELO_LLM, temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model=MODELO_EMBEDDING)

# Ping: si esto imprime el saludo, estas conectado
print(llm.invoke("Responde unicamente: 'Gemini conectado.' y nada mas.").content)

[{'type': 'text', 'text': 'Gemini conectado.', 'extras': {'signature': 'EjQKMgERTTIP8Yqigmc29HizRvyjO7dNAJhYbfMejLTlbBUQPM1aEWXFEJIdRD5dTugTorjA'}}]


## 3. Phoenix primero
Phoenix se inicializa **antes** de los imports de LangChain. ¿Por qué?

La instrumentación funciona "parchando" la librería de LangChain en tiempo de import. Si LangChain ya fue importada y usada, los primeros calls no quedan capturados. Por eso este bloque va **al inicio**.

Tres pasos:

1. **`launch_app()`** — levanta el servidor Phoenix en `localhost:6006`.
2. **`register()`** — crea el tracer apuntando al servidor con un nombre de proyecto.
3. **`LangChainInstrumentor().instrument()`** — a partir de aquí, **cada `invoke` se captura solo**.

> ⚠️ **Si re-ejecutas esta celda** y ves un error de puerto ocupado o de doble instrumentación, reinicia el kernel (`Kernel > Restart`).

In [30]:
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

# 1. Levanta el servidor local de Phoenix
session = px.launch_app()

# 2. Tracer apuntando al servidor local, con el nombre del proyecto
tracer_provider = register(project_name="Proyecto-Final")

# 3. Instrumenta LangChain - guard para que re-ejecutar la celda no rompa
try:
    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
except Exception as e:
    print(f"(Instrumentacion ya activa: {e})")

print(f"\nPhoenix UI:  {session.url}")
print("Abrela en otra pestana. A partir de aqui cada invoke al agente se ve ahi.")

Existing running Phoenix instance detected! Shutting it down and starting a new instance...
2026-07-25 22:29:24.115 Shutting down
2026-07-25 22:29:24.225 Waiting for application shutdown.
2026-07-25 22:29:24.243 Application shutdown complete.
2026-07-25 22:29:24.243 Finished server process [24336]
C:\Users\USUARIO\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value <phoenix.db.types.db_helper_types.Undefined object at 0x0000011D139BC210> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3
Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
OpenTelemetry Tracing Details
|  Phoenix Project: Proyecto-Final
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.


Phoenix UI:  http://localhost:6006/
Abrela en otra pestana. A partir de aqui cada invoke al agente se ve ahi.


## 4. Base de conocimiento - Base documental
El agente de conocimiento no debe inventar las reglas: debe responder a partir del documento oficial. A continuación se carga los tres documentos correspondientes a la base de conocimiento de un agente y debe embeberse de forma independiente.

In [31]:
from pathlib import Path

Documentos = Path("documentos")

catalogo = (Documentos / "01_Catalogo_Productos_Precios.txt").read_text(encoding="utf-8")
politicas = (Documentos / "02_Politicas_Comerciales_Descuentos_Credito.txt").read_text(encoding="utf-8")
crm = (Documentos / "03_Proceso_Ventas_CRM.txt").read_text(encoding="utf-8")

print(f"Catálogo: {len(catalogo):,} caracteres")
print(f"Políticas: {len(politicas):,} caracteres")
print(f"CRM: {len(crm):,} caracteres")

Catálogo: 1,104 caracteres
Políticas: 1,474 caracteres
CRM: 1,168 caracteres


## 5. Chunking + Embeddings + Chroma — indexar la politica

Cortamos la politica en **chunks** (un chunk por seccion numerada), convertimos cada chunk en un **vector** con los embeddings de Gemini y los guardamos en **ChromaDB**, que sabe buscar por similitud.

La mejor estrategia aquí es **un chunk encabezados numerados**: cada encabezado con todos sus párrafos vive en su propio chunk. Esto hace que el embedding capture exactamente el tema de ese artículo y la búsqueda sea precisa.

> 💡 En documentos sin estructura clara (libros, transcripciones, blogs) sí conviene cortar por tamaño con overlap. La estrategia depende siempre de la fuente.

In [32]:
import re
from langchain_chroma import Chroma

def chunkear_por_secciones(texto):
    """ Divide el documento usando encabezados numerados: 1., 2., 3.  """
    cabeceras = list(re.finditer(r"^\d+\.\s", texto, flags=re.MULTILINE))
    chunks = []
    for i, cabecera in enumerate(cabeceras):
        inicio = cabecera.start()
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)
        chunks.append(texto[inicio:fin].strip())
    return chunks
    
# Generar los chunks
chunks_catalogo = chunkear_por_secciones(catalogo)
chunks_politicas = chunkear_por_secciones(politicas)
chunks_crm = chunkear_por_secciones(crm)


# Mostrar un resumen
print("===== RESUMEN DEL CHUNKING =====")

print(f"\nCatálogo: {len(chunks_catalogo)} chunks")
for i, chunk in enumerate(chunks_catalogo, start=1):
    print(f"Chunk {i}: {chunk.splitlines()[0]}")

print(f"\nPolíticas: {len(chunks_politicas)} chunks")
for i, chunk in enumerate(chunks_politicas, start=1):
    print(f"Chunk {i}: {chunk.splitlines()[0]}")

print(f"\nCRM: {len(chunks_crm)} chunks")
for i, chunk in enumerate(chunks_crm, start=1):
    print(f"Chunk {i}: {chunk.splitlines()[0]}")

# VECTOR STORE: CATÁLOGO
vectorstore_catalogo = Chroma.from_texts(
    texts=chunks_catalogo,
    embedding=embeddings,
    metadatas=[{"seccion": i,"fuente": "01_Catalogo_Productos_Precios.txt"} for i in range(len(chunks_catalogo))],
    collection_name="patito_catalogo"
)
# VECTOR : POLÍTICAS
vectorstore_politicas = Chroma.from_texts(
    texts=chunks_politicas,
    embedding=embeddings,
    metadatas=[{"seccion": i,"fuente": "02_Politicas_Comerciales_Descuentos_Credito.txt"}for i in range(len(chunks_politicas))],
    collection_name="patito_politicas"
)
# VECTOR STORE: CRM
vectorstore_crm = Chroma.from_texts(
    texts=chunks_crm,
    embedding=embeddings,
    metadatas=[{"seccion": i,"fuente": "03_Proceso_Ventas_CRM.txt"}for i in range(len(chunks_crm))],
    collection_name="patito_crm"
)
retriever_catalogo = vectorstore_catalogo.as_retriever(search_kwargs={"k": 3})
retriever_politicas = vectorstore_politicas.as_retriever(search_kwargs={"k": 3})
retriever_crm = vectorstore_crm.as_retriever(search_kwargs={"k": 3})
print("Base de conocimiento embebida en Chroma con embeddings de Gemini.")


===== RESUMEN DEL CHUNKING =====

Catálogo: 4 chunks
Chunk 1: 1. LÍNEA PATITO PRO
Chunk 2: 2. LÍNEA PATITO LITE
Chunk 3: 3. ACCESORIOS
Chunk 4: 4. NOTAS

Políticas: 5 chunks
Chunk 1: 1. DESCUENTOS (NIVELES DE AUTORIZACIÓN)
Chunk 2: 2. CONDICIONES DE CRÉDITO
Chunk 3: 3. GARANTÍAS
Chunk 4: 4. DEVOLUCIONES
Chunk 5: 5. ANTICIPOS

CRM: 5 chunks
Chunk 1: 1. ETAPAS DEL EMBUDO (CRM)
Chunk 2: 2. REGISTRO EN EL CRM
Chunk 3: 3. REQUISITOS PARA MARCAR UNA OPORTUNIDAD COMO "GANADA"
Chunk 4: 4. POSVENTA
Chunk 5: 5. BUENAS PRÁCTICAS
Base de conocimiento embebida en Chroma con embeddings de Gemini.


In [33]:
# PRUEBAS DE RECUPERACIÓN DE INFORMACIÓN
def probar_retriever(nombre, retriever, pregunta):
    print("=" * 70)
    print(nombre)
    print("=" * 70)
    print(f"Pregunta: {pregunta}\n")
    documentos = retriever.invoke(pregunta)
    print(f"Documentos recuperados: {len(documentos)}\n")
    for i, doc in enumerate(documentos, start=1):
        print(f"--- Resultado {i} ---")
        print(f"Fuente: {doc.metadata}")
        print(doc.page_content[:400])
        print()
# Prueba catálogo
probar_retriever("RETRIEVER CATÁLOGO", retriever_catalogo, "¿Cuál es el precio del Patito Pro 2026?")
# Prueba políticas
probar_retriever("RETRIEVER POLÍTICAS", retriever_politicas, "¿Qué descuento puede autorizar directamente un vendedor?")
# Prueba CRM
probar_retriever("RETRIEVER CRM", retriever_crm,"¿Qué requisitos se necesitan para marcar una oportunidad como ganada?")

RETRIEVER CATÁLOGO
Pregunta: ¿Cuál es el precio del Patito Pro 2026?

Documentos recuperados: 3

--- Resultado 1 ---
Fuente: {'fuente': '01_Catalogo_Productos_Precios.txt', 'seccion': 0}
1. LÍNEA PATITO PRO
- Patito Pro 2026: precio de lista USD 1,299. Disponibilidad: EN STOCK.
  Características: equipo insignia, procesador de alto rendimiento, 16 GB RAM, 512 GB SSD.
- Patito Pro Max 2026: precio de lista USD 1,799. Disponibilidad: bajo pedido (15 días).

--- Resultado 2 ---
Fuente: {'seccion': 0, 'fuente': '01_Catalogo_Productos_Precios.txt'}
1. LÍNEA PATITO PRO
- Patito Pro 2026: precio de lista USD 1,299. Disponibilidad: EN STOCK.
  Características: equipo insignia, procesador de alto rendimiento, 16 GB RAM, 512 GB SSD.
- Patito Pro Max 2026: precio de lista USD 1,799. Disponibilidad: bajo pedido (15 días).

--- Resultado 3 ---
Fuente: {'seccion': 1, 'fuente': '01_Catalogo_Productos_Precios.txt'}
2. LÍNEA PATITO LITE
- Patito Lite 2026: precio de lista USD 649. Disponibilidad: EN ST

## 🧠 6. Agentes Especializados

### 📗 6.1. Agente Catálogo y Precios
Se crea el agente de **catálogo y precios** mediante una función RAG que recibe una pregunta, consulta la base vectorial correspondiente y utiliza *Gemini* para responder únicamente con la información encontrada.

In [34]:
PROMPT_CATALOGO = """Eres el Agente de Catálogo y Precios de PATITO S.A.
Respondes consultas sobre:
- Productos.
- Precios.
- Disponibilidad.
- Características técnicas.

Reglas:
- Responde únicamente con base en el CONTEXTO.
- No inventes información.
- Si la respuesta no aparece en el contexto responde exactamente:
"No tengo esa informacio en el catálogo actual."
- Sé breve y directo.
"""

def responder_catalogo(pregunta: str) -> str:
    """Pipeline RAG del catálogo."""
    docs = retriever_catalogo.invoke(pregunta)
    contexto = "\n\n---\n\n".join(
        doc.page_content
        for doc in docs
    )
    respuesta = llm.invoke([
        {"role": "system","content": PROMPT_CATALOGO},
        {"role": "user","content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    return respuesta.content

# Prueba
print(responder_catalogo("Cual es el monto maximo de un almuerzo de trabajo?"))

[{'type': 'text', 'text': 'No tengo esa informacio en el catálogo actual.', 'extras': {'signature': 'EjQKMgERTTIPFqt5fJPzPYeJ17bvFcxT6xa+jxO+X/8lD828UTK92gjjUmWTIxeKkP7uXSHz'}}]


### ⚖️ 6.2. Agente de Políticas Comerciales
Se implementa la función RAG para el agente de Políticas Comerciales, aplicando un prompt estricto que restringe la respuesta a las normas explícitas de la empresa para evitar alucinaciones.

In [35]:
prompt_politicas = ("""Eres el Agente de Políticas Comerciales de PATITO S.A.

Tu función es responder únicamente consultas relacionadas con:
- descuentos;
- niveles de autorización;
- condiciones de crédito;
- garantías;
- devoluciones;
- anticipos.

Reglas:
- Utiliza exclusivamente el contexto proporcionado.
- No inventes políticas ni condiciones comerciales.
- Si la información no existe en las políticas responde:
  "No dispongo de esa información en las políticas comerciales actuales."
- Responde de forma clara y profesional.
"""
)

def responder_politicas(pregunta:str) -> str:
    """Pipeline RAG del catálogo"""
    docs = retriever_politicas.invoke(pregunta)
    contexto = "\n\n---\n\n".join(
        doc.page_content
        for doc in docs
    )
    
    respuesta = llm.invoke([
        {"role": "system","content": prompt_politicas},
        {"role": "user","content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    return respuesta.content
# Prueba descuento permitido
print(responder_politicas("¿Qué porcentaje de descuento puede autorizar directamente un vendedor?"))
# Prueba crédito
print(responder_politicas("¿Qué requisitos existen para vender a crédito?"))
# Prueba control de alucinación
print (responder_politicas("¿Patito S.A. ofrece descuentos del 80%?"))

[{'type': 'text', 'text': 'Un vendedor puede autorizar directamente un descuento de hasta el 10%, sin necesidad de aprobación adicional.', 'extras': {'signature': 'EjQKMgERTTIPM8UPdFBLV9s0begtlVjloBpsMc+2cmWdA5eBrVoaRZ0DNKA+j7kXf80d7vM4'}}]
[{'type': 'text', 'text': 'Para realizar ventas a crédito en PATITO S.A., se deben cumplir los siguientes requisitos:\n\n*   Presentar la solicitud de crédito.\n*   Entregar los documentos de la empresa.\n*   Obtener la aprobación del área financiera.\n\nCabe señalar que, para clientes nuevos, la primera compra suele ser de contado y el crédito se evalúa tras la primera operación y un análisis de crédito. Asimismo, las ventas a crédito sin una línea aprobada requieren la autorización de la gerencia.', 'extras': {'signature': 'EjQKMgERTTIPQDhpFWatzNmaZ68ANIEAx7ndrjciwf9466f54Z9RbCAAxydBcizgWOLQOXkC'}}]
[{'type': 'text', 'text': 'De acuerdo con las políticas comerciales de PATITO S.A., los descuentos superiores al 30% no están permitidos, salvo en cas

### 💸 6.3. Agente Proceso de Ventas y CRM
Se configura el agente de CRM mediante un pipeline RAG encargado de resolver dudas operativas sobre el pipeline de ventas y las etapas del proceso comercial.

In [36]:
prompt_crm = ("""Eres el Agente de Proceso de Ventas y CRM de PATITO S.A.
Tu función es responder únicamente consultas relacionadas con:
- etapas del proceso comercial;
- registro de oportunidades;
- requisitos del CRM;
- cierre de ventas;
- seguimiento posventa.

Reglas:
- Utiliza solamente la información del contexto recuperado.
- No inventes procesos, campos del CRM ni requisitos.
- Si la información no aparece en el manual responde:
  "No dispongo de esa información en el proceso de ventas y CRM actual."
- Mantén respuestas claras y profesionales.
"""
)
def responder_crm(pregunta:str) -> str:
    """Pipeline RAG del catálogo"""
    docs = retriever_crm.invoke(pregunta)
    contexto = "\n\n---\n\n".join(
        doc.page_content
        for doc in docs
    )
    
    respuesta = llm.invoke([
        {"role": "system","content": prompt_crm},
        {"role": "user","content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    return respuesta.content

# Prueba 1 etapas del embudo
print(responder_crm("¿Cuáles son las etapas del embudo de ventas?"))
#Prueba 2 oportunidad ganada
print(responder_crm("¿Qué requisitos se necesitan para marcar una oportunidad como ganada?"))
#Prueba 3 pregunta fuera de dominio
print(responder_crm("¿Cuál es el precio del Patito Pro 2026?"))

[{'type': 'text', 'text': 'Las etapas del embudo de ventas en el CRM de PATITO S.A. son las siguientes:\n\n1. Prospecto\n2. Contacto\n3. Calificación\n4. Propuesta/Cotización\n5. Negociación\n6. Cierre (Ganada o Perdida)', 'extras': {'signature': 'EjQKMgERTTIPOX0ljpqGIboI4E7BWY4b+N5DlB92aIfaWqI24w3N2ow2h+f/mPSYpIPS1jAl'}}]
[{'type': 'text', 'text': 'Para marcar una oportunidad como "ganada" en el CRM de PATITO S.A., es necesario registrar la siguiente información:\n\n*   Orden de compra o contrato firmado por el cliente.\n*   Datos de facturación completos del cliente.\n*   Productos, cantidades y precios finales (incluyendo el descuento aplicado y su autorización).\n*   Condición de pago (contado o crédito) y plazo acordado.\n*   Monto total de la venta y fecha de cierre.\n*   Fecha de entrega comprometida.', 'extras': {'signature': 'EjQKMgERTTIPlWt7fdPoM8FoO4e0acS0komibwGV/vpWQElBxFdQu9+RK7FYacwetW1ljS38'}}]
[{'type': 'text', 'text': 'No dispongo de esa información en el proceso de v

### 🖼️ 6.4. Agente Multimodal de imagen

Gemini es multimodal: puede recibir una imagen y leerla. 
Genera sintéticamente una imagen local *(patito_pro.png)* usando la librería Pillow para simular una ficha técnica de un producto con especificaciones visuales.

Codifica la imagen creada en formato Base64 y la envía a Gemini mediante mensajes multimodales para extraer de forma estructurada los datos del producto.

In [37]:
from PIL import Image, ImageDraw
import base64

def crear_ficha_producto_demo(ruta="patito_pro.png"):
    """Genera una imagen simple de un producto de PATITO S.A."""

    img = Image.new("RGB", (500, 350), "white")
    d = ImageDraw.Draw(img)

    lineas = [
        "PATITO S.A.",
        "CATÁLOGO DE PRODUCTOS",
        "--------------------------------",
        "Producto: Patito Pro 2026",
        "Precio: USD 1299",
        "Disponibilidad: EN STOCK",
        "Garantía: 12 meses",
        "RAM: 16 GB",
        "SSD: 512 GB"
    ]
    y = 20
    for linea in lineas:
        d.text((20, y), linea, fill="black")
        y += 32

    img.save(ruta)

    return ruta

ruta_imagen = crear_ficha_producto_demo()

print("Imagen creada:", ruta_imagen)

Imagen creada: patito_pro.png


In [38]:
from langchain_core.messages import HumanMessage

def analizar_comprobante(ruta_imagen: str) -> str:
    """Agente multimodal: envia la imagen a Gemini (vision) y extrae los datos del comprobante."""
    try:
        with open(ruta_imagen, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
    except FileNotFoundError:
        return f"No se encontro la imagen '{ruta_imagen}'."

    prompt = (
        "Analiza esta imagen de un producto de PATITO S.A. "
        "Extrae y devuelve en líneas separadas: producto, precio, disponibilidad, "
        "garantía y características visibles. "
        "Si algún dato no aparece, escribe 'no visible'. "
        "No inventes información."
    )
    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": f"data:image/png;base64,{b64}"},
    ])
    return llm.invoke([msg]).content

# Prueba
print(analizar_comprobante("patito_pro.png"))

[{'type': 'text', 'text': 'Producto: Patito Pro 2026\nPrecio: USD 1299\nDisponibilidad: EN STOCK\nGarantía: 12 meses\nCaracterísticas visibles: RAM: 16 GB, SSD: 512 GB', 'extras': {'signature': 'EjQKMgERTTIPOxe4uDX1xAu6x0ooSLkIfP5GaQ3VDQs3ZjBD4qZIS8jU+PU96XiGunfF36Zq'}}]


## ✍️ 7. Agente de Acción (Registro) - Sistema de control

Se construye la herramienta *@tool registrar_oportunidad*, la cual valida campos requeridos, exige bandera de confirmación, aplica reglas de descuento y guarda las oportunidades en un archivo local .txt.

In [39]:
from langchain.tools import tool
from datetime import datetime
from pathlib import Path

REGISTRO_PATH = "registro_oportunidades.txt"
CAMPOS_OBLIGATORIOS = ["cliente", "contacto", "producto", "cantidad", "precio_unitario", "descuento_aplicado", "condicion_pago", "monto_total",]

def _siguiente_id():
    if not Path(REGISTRO_PATH).exists():
        return "OP-0001"
    n = sum(1 for l in open(REGISTRO_PATH, encoding="utf-8") if l.strip())
    return f"OP-{n + 1:04d}"

@tool
def registrar_oportunidad(cliente: str = "", contacto: str = "", producto: str = "", cantidad: int = 0, precio_unitario: float = 0.0,
                          descuento_aplicado: float = 0.0, autorizacion_descuento: str = "", condicion_pago: str = "", monto_total: float = 0.0,
                          confirmar: bool = False,) -> str:
    """ Registra una oportunidad comercial en un archivo de texto. Requiere:cliente, contacto, producto, cantidad, precio_unitario,
    descuento_aplicado, condicion_pago y monto_total. Si el descuento supera 10%, requiere autorizacion_descuento. Si falta información, 
    no registra y devuelve los datos faltantes. Si confirmar=False, solo muestra el resumen y solicita confirmación. """
    datos = {"cliente": cliente, "contacto": contacto, "producto": producto, "cantidad": cantidad, "precio_unitario": precio_unitario,
             "descuento_aplicado": descuento_aplicado, "autorizacion_descuento": autorizacion_descuento, "condicion_pago": condicion_pago,
             "monto_total": monto_total}

    faltantes = [
        k for k in CAMPOS_OBLIGATORIOS
        if not str(datos[k]).strip() or (k in {"cantidad", "precio_unitario", "monto_total"} and float(datos[k]) <= 0)
    ]

    if float(descuento_aplicado) > 10 and not str(autorizacion_descuento).strip():
        faltantes.append("autorizacion_descuento")

    faltantes = sorted(set(faltantes))
    if faltantes:
        return "No se registró la oportunidad. Faltan datos obligatorios: " + ", ".join(faltantes) + "."

    resumen = (
        "Resumen de la oportunidad:\n" 
        f"- Cliente: {cliente}\n"
        f"- Contacto: {contacto}\n"
        f"- Producto: {producto}\n"
        f"- Cantidad: {cantidad}\n"
        f"- Precio unitario: USD {float(precio_unitario):.2f}\n"
        f"- Descuento aplicado: {float(descuento_aplicado):.2f}%\n"
        f"- Autorización descuento: {autorizacion_descuento or 'No aplica'}\n"
        f"- Condición de pago: {condicion_pago}\n"
        f"- Monto total: USD {float(monto_total):.2f}\n"
    )

    if not confirmar:
        return resumen + "\nConfirma la operación invocando nuevamente la tool con confirmar=True."

    rid = _siguiente_id()
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    registro = (
    f"ID: {rid}\n"
    f"Fecha: {ts}\n"
    f"Cliente: {cliente}\n"
    f"Producto: {producto}\n"
    f"Cantidad: {cantidad}\n"
    f"Monto estimado: USD {float(monto_total)}\n"
    f"Etapa: Prospecto\n"
    f"Estado: Abierta\n")
    
    with open(REGISTRO_PATH, "a", encoding="utf-8") as f:
        f.write(registro + "------------------------------\n")
    
    return registro

# Prueba 1: faltan datos
print(registrar_oportunidad.invoke({"cliente": "Comercial ABC", "producto": "Patito Pro 2026", "cantidad": 10}))
# Prueba 2: datos completos, pero sin confirmar
print(registrar_oportunidad.invoke({"cliente": "Comercial ABC", "contacto": "María López", "producto": "Patito Pro 2026", "cantidad": 10, "precio_unitario": 1299, "descuento_aplicado": 8, "condicion_pago": "Contado", "monto_total": 11990}))
# Prueba 3: confirmación y registro
print(registrar_oportunidad.invoke({"cliente": "Comercial ABC", "contacto": "María López", "producto": "Patito Pro 2026", "cantidad": 10, "precio_unitario": 1299, "descuento_aplicado": 8, "condicion_pago": "Contado", "monto_total": 11990, "confirmar": True}))

No se registró la oportunidad. Faltan datos obligatorios: condicion_pago, contacto, monto_total, precio_unitario.
Resumen de la oportunidad:
- Cliente: Comercial ABC
- Contacto: María López
- Producto: Patito Pro 2026
- Cantidad: 10
- Precio unitario: USD 1299.00
- Descuento aplicado: 8.00%
- Autorización descuento: No aplica
- Condición de pago: Contado
- Monto total: USD 11990.00

Confirma la operación invocando nuevamente la tool con confirmar=True.
ID: OP-0001
Fecha: 2026-07-25 22:31:27
Cliente: Comercial ABC
Producto: Patito Pro 2026
Cantidad: 10
Monto estimado: USD 11990.0
Etapa: Prospecto
Estado: Abierta



## 8. Orquestador de Agentes
Se ensambla el agente orquestador principal encapsulando los agentes especializados y la herramienta de registro dentro de funciones *@tool*, añadiendo memoria persistente con *InMemorySaver*. Exponemos cada capacidad como una **tool** y construimos el orquestador con `create_agent` (LangChain 1.x). El orquestador **decide y enruta** a cualquiera de los cinco agentes:

- `consultar_catalogo` → 📗 Agente Catálogo y Precios
- `consultar_politicas` → ⚖️ Agente de politicas comerciales
- `consultar_crm` → 💸 Agente de proceso de ventas y crm
- `analizar_producto_tool` → 🖼️ Agente Multimodal
- `registrar_oportunidad` → ✍️ Agente de Accion

Ademas le damos **memoria** (`InMemorySaver` + `thread_id`), asi el orquestador recuerda la conversacion: si al registrar le falta un dato, lo pide y, cuando se lo das en el siguiente mensaje, **completa el registro**.


In [40]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
import uuid

@tool
def consultar_catalogo(pregunta: str) -> str:
    """Responde preguntas sobre productos, precios, disponibilidad y características."""
    return responder_catalogo(pregunta)

@tool
def consultar_politicas(pregunta: str) -> str:
    """Responde preguntas sobre descuentos, crédito, garantías, devoluciones y anticipos."""
    return responder_politicas(pregunta)
    
@tool
def consultar_crm(pregunta: str) -> str:
    """Responde preguntas sobre etapas del embudo, CRM, cierre de ventas y posventa."""
    return responder_crm(pregunta)

@tool
def analizar_producto_tool(ruta_imagen: str) -> str:
    """Analiza una imagen de producto o ficha técnica."""
    return analizar_producto(ruta_imagen)

tools_orquestador = [consultar_catalogo, consultar_politicas, consultar_crm, registrar_oportunidad, analizar_producto_tool]

SYSTEM_PROMPT = """Eres el orquestador de PATITO S.A. Coordinas agentes especializados construidos con LangChain.

Agentes disponibles:
- consultar_catalogo: productos, precios, disponibilidad y características.
- consultar_politicas: descuentos, crédito, garantías, devoluciones y anticipos.
- consultar_crm: etapas del embudo, registro en CRM, cierre y posventa.
- registrar_oportunidad: registrar o guardar una oportunidad comercial en el archivo de texto.

Reglas de ruteo:
- Si la consulta es sobre precios, productos o disponibilidad, usa consultar_catalogo.
- Si la consulta es sobre descuentos, crédito, garantías, devoluciones o anticipos, usa consultar_politicas.
- Si la consulta es sobre etapas, CRM, cierre o posventa, usa consultar_crm.
- Si el usuario pide registrar, guardar o crear una oportunidad, usa registrar_oportunidad.
- Si faltan datos obligatorios para registrar, pide los datos faltantes y no registres todavía.
- Si la consulta es mixta, usa uno o más agentes y consolida la respuesta.
- Indica siempre qué agente(s) usaste y qué fuente(s) consultaste.
- No inventes información.
- Si no hay información suficiente, dilo explícitamente.

Si se agregara una tool multimodal, usa esa tool únicamente cuando el usuario entregue una imagen o una ruta de imagen.
"""
memoria = InMemorySaver()

orquestador = create_agent(
    model=llm,
    tools=tools_orquestador,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memoria,
)

print("Tools registradas en el orquestador:")
for t in tools_orquestador:
    print("  -", t.name)

def _imprimir_pasos(resultado):
    for m in resultado["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print(f"[TOOL] {tc['name']}({tc['args']})")
        if m.__class__.__name__ == "ToolMessage":
            print(f"[RESPONSE] {str(m.content)[:300]}\n")

def extraer_texto(content):
    """Devuelve solo texto plano si Gemini retorna bloques en lista."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        partes = []
        for b in content:
            if isinstance(b, dict):
                partes.append(b.get("text", ""))
            elif isinstance(b, str):
                partes.append(b)
        return "".join(partes).strip()
    return str(content)

def consultar(pregunta: str, thread_id: str = None):
    """Invoca al orquestador y muestra tools + respuesta final."""
    thread_id = thread_id or f"patito-{uuid.uuid4().hex[:8]}"
    config = {"configurable": {"thread_id": thread_id}}

    print(f">>> Usuario: {pregunta}\n")
    resultado = orquestador.invoke({"messages": [{"role": "user", "content": pregunta}]},config)
    _imprimir_pasos(resultado)
    print("=== Respuesta final ===")
    print(extraer_texto(resultado["messages"][-1].content))
    return resultado

Tools registradas en el orquestador:
  - consultar_catalogo
  - consultar_politicas
  - consultar_crm
  - registrar_oportunidad
  - analizar_producto_tool


### 9.1. Prueba - (Agente Catálogo y Precios)
Se ejecuta una consulta sobre precios de productos para validar que el orquestador identifique la intención y delegue la respuesta a la herramienta del Catálogo.

In [41]:
consultar("¿Cuál es el precio del Patito Pro 2026?")

>>> Usuario: ¿Cuál es el precio del Patito Pro 2026?

[TOOL] consultar_catalogo({'pregunta': '¿Cuál es el precio del Patito Pro 2026?'})
[RESPONSE] [{'type': 'text', 'text': 'El precio del Patito Pro 2026 es USD 1,299.', 'extras': {'signature': 'EjQKMgERTTIPFD+NoBVVY6A73E5fwLxV+LQsbWCv9s55VY9sBcuCpTdm7CNvhJLrbDdhzY/F'}}]

=== Respuesta final ===
El precio del Patito Pro 2026 es de USD 1,299.

Esta información fue obtenida a través del agente **consultar_catalogo**, consultando la base de datos de productos de PATITO S.A.


{'messages': [HumanMessage(content='¿Cuál es el precio del Patito Pro 2026?', additional_kwargs={}, response_metadata={}, id='f9a66a52-6c46-46f5-a19b-5d2844b03f21'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_catalogo', 'arguments': '{"pregunta": "\\u00bfCu\\u00e1l es el precio del Patito Pro 2026?"}'}, '__gemini_function_call_thought_signatures__': {'o3vIPQjs': 'EjQKMgERTTIP/aAa1UroAanngDD0GpsQOJWJ9zvkZL5CVoER2hb3gRg85HYALtaw5JcFrz6R'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f9c7a-c7ed-7501-a79c-71b4985e3e72-0', tool_calls=[{'name': 'consultar_catalogo', 'args': {'pregunta': '¿Cuál es el precio del Patito Pro 2026?'}, 'id': 'o3vIPQjs', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 759, 'output_tokens': 33, 'total_tokens': 792, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content=[{'type': 

### 9.2. Prueba - (Agente de de Políticas Comerciales)
Se envía una consulta sobre condiciones comerciales para comprobar que el orquestador redirija el flujo hacia el agente de Políticas.

In [42]:
consultar("¿Cuál es el descuento máximo que puede autorizar un vendedor?")

>>> Usuario: ¿Cuál es el descuento máximo que puede autorizar un vendedor?

[TOOL] consultar_politicas({'pregunta': '¿Cuál es el descuento máximo que puede autorizar un vendedor?'})
[RESPONSE] [{'type': 'text', 'text': 'El descuento máximo que un vendedor puede autorizar directamente, sin necesidad de aprobación adicional, es del 10%.', 'extras': {'signature': 'EjQKMgERTTIP0jIhgn0RH0P4kccJAWD//VMMZDIawGazJKpVCrjpN22VJ7bPPXqVGfczsXKZ'}}]

=== Respuesta final ===
Para responder a tu consulta, he consultado al agente **consultar_politicas**.

De acuerdo con las políticas de PATITO S.A., el descuento máximo que un vendedor puede autorizar directamente, sin necesidad de aprobación adicional, es del **10%**.


{'messages': [HumanMessage(content='¿Cuál es el descuento máximo que puede autorizar un vendedor?', additional_kwargs={}, response_metadata={}, id='21b62e5f-d3b8-473b-a544-d9ca17efa82a'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_politicas', 'arguments': '{"pregunta": "\\u00bfCu\\u00e1l es el descuento m\\u00e1ximo que puede autorizar un vendedor?"}'}, '__gemini_function_call_thought_signatures__': {'x5yy5oii': 'EjQKMgERTTIPv0gKezxmR5ysEmKodA+4FeJEW0nqoilMwDlB00MouN1YUfoRgApVc7/ZOOFp'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f9c7a-e512-7973-bbaa-f021e8b26243-0', tool_calls=[{'name': 'consultar_politicas', 'args': {'pregunta': '¿Cuál es el descuento máximo que puede autorizar un vendedor?'}, 'id': 'x5yy5oii', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 757, 'output_tokens': 31, 'total_tokens': 788, 'i

### 9.3. Prueba - (Agente de Proceso de Ventas y CRM)
Se realiza una pregunta sobre las etapas de venta para evaluar la correcta delegación hacia el agente experto en CRM.

In [43]:
consultar("¿Qué requisitos se necesitan para marcar una oportunidad como ganada?")

>>> Usuario: ¿Qué requisitos se necesitan para marcar una oportunidad como ganada?

[TOOL] consultar_crm({'pregunta': '¿Qué requisitos se necesitan para marcar una oportunidad como ganada?'})
[RESPONSE] [{'type': 'text', 'text': 'Para marcar una oportunidad como "ganada" en el CRM de PATITO S.A., es necesario registrar la siguiente información:\n\n*   Orden de compra o contrato firmado por el cliente.\n*   Datos de facturación completos del cliente.\n*   Productos, cantidades y precios finales (inc

=== Respuesta final ===
Para marcar una oportunidad como "ganada" en el CRM de PATITO S.A., se requiere contar con la siguiente información y documentación:

*   **Orden de compra o contrato** firmado por el cliente.
*   **Datos de facturación** completos del cliente.
*   **Detalle de la venta:** productos, cantidades y precios finales (incluyendo el descuento aplicado y su respectiva autorización, si aplica).
*   **Condición de pago:** especificar si es de contado o crédito, junto con el p

{'messages': [HumanMessage(content='¿Qué requisitos se necesitan para marcar una oportunidad como ganada?', additional_kwargs={}, response_metadata={}, id='52572992-4ded-47b7-a353-04a55b4cfbe5'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_crm', 'arguments': '{"pregunta": "\\u00bfQu\\u00e9 requisitos se necesitan para marcar una oportunidad como ganada?"}'}, '__gemini_function_call_thought_signatures__': {'xavdQW5J': 'EjQKMgERTTIPg9UxpLFTIT6T93xblAsDKF7iYVce1/sQ7EjlOAdhWYlaSxiZF5OQHWmrfE5Y'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f9c7a-fa21-7c62-a93f-e4d0898e4c68-0', tool_calls=[{'name': 'consultar_crm', 'args': {'pregunta': '¿Qué requisitos se necesitan para marcar una oportunidad como ganada?'}, 'id': 'xavdQW5J', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 757, 'output_tokens': 30, 'total_tokens': 7

### 9.4. Ver el registro generado
Se abre y lee el archivo de texto registro_oportunidades.txt para asegurar que los eventos de escritura en disco se hayan guardado con el formato correcto.

In [44]:
print(Path(REGISTRO_PATH).read_text(encoding="utf-8") if Path(REGISTRO_PATH).exists() else "Aun no hay registros.")

ID: OP-0001
Fecha: 2026-07-25 22:31:27
Cliente: Comercial ABC
Producto: Patito Pro 2026
Cantidad: 10
Monto estimado: USD 11990.0
Etapa: Prospecto
Estado: Abierta
------------------------------



## 10. Generación de trazas

In [45]:
consultar("¿Cuál es el precio del Patito Pro 2026?")

>>> Usuario: ¿Cuál es el precio del Patito Pro 2026?

[TOOL] consultar_catalogo({'pregunta': '¿Cuál es el precio del Patito Pro 2026?'})
[RESPONSE] [{'type': 'text', 'text': 'El precio del Patito Pro 2026 es USD 1,299.', 'extras': {'signature': 'EjQKMgERTTIPPmC1AHVl58w4UIyeFFQi7VFe9XZxecC2xJflXtxouyBUgwl3fJUsx097S4tl'}}]

=== Respuesta final ===
El precio del Patito Pro 2026 es de USD 1,299.

Esta información fue obtenida a través del agente **consultar_catalogo**, consultando la base de datos de productos y precios de PATITO S.A.


{'messages': [HumanMessage(content='¿Cuál es el precio del Patito Pro 2026?', additional_kwargs={}, response_metadata={}, id='7f82cd6f-058b-457d-8cd2-78874dd4ea45'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_catalogo', 'arguments': '{"pregunta": "\\u00bfCu\\u00e1l es el precio del Patito Pro 2026?"}'}, '__gemini_function_call_thought_signatures__': {'cK35fsBZ': 'EjQKMgERTTIPY5gHhM35WqXDBHdJH0cCuN2b59nV5DKqJeR7fRdBWZWeygzTtteL53GBTWcP'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f9c7b-4c31-73c2-aaf1-6ef73bcdcdab-0', tool_calls=[{'name': 'consultar_catalogo', 'args': {'pregunta': '¿Cuál es el precio del Patito Pro 2026?'}, 'id': 'cK35fsBZ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 759, 'output_tokens': 33, 'total_tokens': 792, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content=[{'type': 

In [20]:
consultar("¿Qué descuento puede autorizar un vendedor?")

>>> Usuario: ¿Qué descuento puede autorizar un vendedor?

[TOOL] consultar_politicas({'pregunta': '¿Qué descuento puede autorizar un vendedor?'})
[RESPONSE] [{'type': 'text', 'text': 'Un vendedor puede autorizar directamente descuentos de hasta el 10%, sin necesidad de aprobación adicional.', 'extras': {'signature': 'EjQKMgERTTIP7ybwlFSk15u6FZJ71pYL+YdNEBhOVjQs2aDbY/bRRNOGv6xsVtgvMV6gypsS'}}]

=== Respuesta final ===
Para responder a tu consulta, he consultado el agente **consultar_politicas**.

De acuerdo con nuestras políticas, un vendedor puede autorizar directamente descuentos de hasta el **10%** sin necesidad de aprobación adicional. Si el descuento supera este porcentaje, será necesario solicitar una autorización especial.


{'messages': [HumanMessage(content='¿Qué descuento puede autorizar un vendedor?', additional_kwargs={}, response_metadata={}, id='f0fbf1e8-7e07-4642-9e41-7176be13331f'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_politicas', 'arguments': '{"pregunta": "\\u00bfQu\\u00e9 descuento puede autorizar un vendedor?"}'}, '__gemini_function_call_thought_signatures__': {'k2Fqfiui': 'EjQKMgERTTIPpemDN0CJlHW8qmET4mqkjh7QMjIi63MpS6vYwJaQoKtamZoCZJF0RqRCQaT0'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f9c75-9d90-75d2-bbea-c9514f8596eb-0', tool_calls=[{'name': 'consultar_politicas', 'args': {'pregunta': '¿Qué descuento puede autorizar un vendedor?'}, 'id': 'k2Fqfiui', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 753, 'output_tokens': 27, 'total_tokens': 780, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(cont

In [21]:
consultar("¿Qué requisitos se necesitan para marcar una oportunidad como ganada?")

>>> Usuario: ¿Qué requisitos se necesitan para marcar una oportunidad como ganada?

[TOOL] consultar_crm({'pregunta': '¿Qué requisitos se necesitan para marcar una oportunidad como ganada?'})
[RESPONSE] [{'type': 'text', 'text': 'Para marcar una oportunidad como "ganada" en el CRM, es obligatorio registrar los siguientes requisitos:\n\n*   Orden de compra o contrato firmado por el cliente.\n*   Datos de facturación completos del cliente.\n*   Productos, cantidades y precios finales (incluyendo el d

=== Respuesta final ===
Para marcar una oportunidad como "ganada" en el CRM, es necesario cumplir con los siguientes requisitos:

*   **Documentación:** Orden de compra o contrato firmado por el cliente.
*   **Información del cliente:** Datos de facturación completos.
*   **Detalle de la venta:** Productos, cantidades y precios finales (incluyendo el descuento aplicado y su autorización).
*   **Condiciones comerciales:** Condición de pago (contado o crédito) y plazo acordado.
*   **Montos y

{'messages': [HumanMessage(content='¿Qué requisitos se necesitan para marcar una oportunidad como ganada?', additional_kwargs={}, response_metadata={}, id='3339291c-fb7c-4280-9a5f-3baf518db207'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_crm', 'arguments': '{"pregunta": "\\u00bfQu\\u00e9 requisitos se necesitan para marcar una oportunidad como ganada?"}'}, '__gemini_function_call_thought_signatures__': {'rx5E0OKt': 'EjQKMgERTTIPojFH9/kIO7pY3KQtjiHNh9WXi+juMuCpAh3Z91eFo0v3YPytTI+GKpxBYD2/'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f9c75-a744-7100-a962-67f61c0c351f-0', tool_calls=[{'name': 'consultar_crm', 'args': {'pregunta': '¿Qué requisitos se necesitan para marcar una oportunidad como ganada?'}, 'id': 'rx5E0OKt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 757, 'output_tokens': 30, 'total_tokens': 7

In [46]:
consultar("Registrar una oportunidad para Comercial ABC, 10 unidades de Patito Pro 2026, 8% de descuento, pago de contado.")

>>> Usuario: Registrar una oportunidad para Comercial ABC, 10 unidades de Patito Pro 2026, 8% de descuento, pago de contado.

[TOOL] consultar_catalogo({'pregunta': '¿Cuál es el precio unitario del producto Patito Pro 2026?'})
[RESPONSE] [{'type': 'text', 'text': 'El precio del Patito Pro 2026 es USD 1,299.', 'extras': {'signature': 'EjQKMgERTTIP5gz0qcukG0zlx2XFUzqQyWO98Iv5qK4OorPBxPk82FgbG7mHL1Si5EjPcoZc'}}]

[TOOL] registrar_oportunidad({'cliente': 'Comercial ABC', 'confirmar': False, 'condicion_pago': 'contado', 'cantidad': 10, 'producto': 'Patito Pro 2026', 'precio_unitario': 1299, 'descuento_aplicado': 8})
[RESPONSE] No se registró la oportunidad. Faltan datos obligatorios: contacto, monto_total.

=== Respuesta final ===
Para proceder con el registro de la oportunidad para **Comercial ABC**, he consultado el catálogo y el sistema de registro.

**Estado del proceso:**
*   **Precio unitario:** USD 1,299 (obtenido mediante `consultar_catalogo`).
*   **Cálculo:** 10 unidades a USD 1,2

{'messages': [HumanMessage(content='Registrar una oportunidad para Comercial ABC, 10 unidades de Patito Pro 2026, 8% de descuento, pago de contado.', additional_kwargs={}, response_metadata={}, id='8edeef0d-6dfe-4423-ac2f-36ec02fc6841'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_catalogo', 'arguments': '{"pregunta": "\\u00bfCu\\u00e1l es el precio unitario del producto Patito Pro 2026?"}'}, '__gemini_function_call_thought_signatures__': {'a9LqsYLo': 'EjQKMgERTTIPfKzMMgqnvBNgLt15N18p20DwE7zIHxJU3ai1KPSL0JMfusj6/dd012SbS/W/'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f9c7b-9162-7b70-b336-cfd7c8bdb288-0', tool_calls=[{'name': 'consultar_catalogo', 'args': {'pregunta': '¿Cuál es el precio unitario del producto Patito Pro 2026?'}, 'id': 'a9LqsYLo', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 776, 'output_to

# 15. Interfaz del Asistente

Interfaz conversacional que permite al usuario interactuar con el orquestador multiagente de **Patito S.A**. A través del chat, las consultas son dirigidas hacia los agentes especializados de *catálogo*, *políticas comerciales* y *CRM*, quienes utilizan **herramientas+RAG** conectadas a las bases de conocimiento para generar respuestas contextualizadas. La interfaz incorpora historial de conversación, manejo de errores y ejecución no bloqueante dentro de Jupyter.

In [58]:
import threading
import uuid
import traceback
import html as html_lib

import ipywidgets as widgets
from IPython.display import display

# CONFIGURACIÓN DE MEMORIA
config_patito = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    }
}

historial_patito = []

out_debug = widgets.Output()

# CSS

CSS_CHAT = """

<style>

.patito-card {
    background:white;
    border-radius:12px;
    padding:16px;
    font-family:Arial, sans-serif;
    box-shadow:0 2px 10px rgba(0,0,0,0.1);
    max-width:700px;
    margin:auto;
}

.patito-header {
    background:#162447;
    color:white;
    padding:14px;
    border-radius:10px 10px 0 0;
    font-weight:bold;
}

.patito-chat-box {
    height:420px;
    overflow-y:auto;
    background:#fafafa;
    border:1px solid #ddd;
    border-radius:8px;
    padding:12px;
    display:flex;
    flex-direction:column-reverse;
}

.patito-row {
    display:flex;
    margin:8px 0;

}
.patito-row.user {
    justify-content:flex-end;
}

.patito-row.agent {
    justify-content:flex-start;
}

.patito-msg {
    max-width:80%;
    padding:10px 14px;
    border-radius:14px;
    line-height:1.4;
}

.user .patito-msg {
    background:#162447;
    color:white;
}

.agent .patito-msg {
    background:white;
    border:1px solid #ddd;
    color:#333;
}

.patito-vacio {
    text-align:center;
    color:#999;
    padding:60px;
}

button {
    border-radius:6px !important;
}

</style>

"""
# WIDGETS

box_chat = widgets.HTML(
    value="""
    <div class="patito-chat-box">
        <div class="patito-vacio">
            🤖 Asistente Patito S.A. listo.
        </div>
    </div>
    """
)

txt_input = widgets.Textarea(
    placeholder="Escribe tu petición...",
    layout=widgets.Layout(
        width="100%",
        height="60px"
    )
)

btn_enviar = widgets.Button(
    description="Enviar",
    icon="paper-plane",
    button_style="primary",
    layout=widgets.Layout(
        width="100%"
    )
)

btn_nuevo = widgets.Button(
    description="Nuevo chat",
    icon="refresh",
    layout=widgets.Layout(
        width="100%"
    )
)

lbl_estado = widgets.HTML(value="")

# RENDER DEL CHAT
def render_mensajes():
    if not historial_patito:
        return """
        <div class="patito-chat-box">
            <div class="patito-vacio">
            🤖 Asistente Patito S.A. listo.
            </div>
        </div>
        """

    html = [ '<div class="patito-chat-box">' ]
    for mensaje in reversed(historial_patito):
        tipo = mensaje["rol"]
        nombre = (
            "👤 Tú"
            if tipo == "user"
            else "🤖 Asistente"
        )

        texto = (html_lib.escape(mensaje["texto"]).replace("\n","<br>"))


        html.append(f"""
            <div class="patito-row {tipo}">
                <div class="patito-msg">
                    <b>{nombre}</b>
                    <br>
                    {texto}
                </div>
            </div>
            """
        )

    html.append("</div>")


    return "".join(html)



# =====================================================
# CONSULTAR ORQUESTADOR
# =====================================================

def ejecutar_orquestador(pregunta):

    global config_patito


    try:

        resultado = orquestador.invoke(

            {
                "messages":[
                    {
                        "role":"user",
                        "content":pregunta
                    }
                ]
            },

            config_patito

        )



        respuesta = extraer_texto(
            resultado["messages"][-1].content
        )



        herramientas = []


        for mensaje in resultado["messages"]:

            for tool in (
                getattr(
                    mensaje,
                    "tool_calls",
                    []
                )
                or []
            ):

                herramientas.append(
                    tool["name"]
                )



        if herramientas:

            respuesta += (

                "\n\n🔧 Tools utilizadas: "
                +
                ", ".join(
                    herramientas
                )

            )



        historial_patito.append(

            {
                "rol":"agent",
                "texto":respuesta
            }

        )


    except Exception:


        historial_patito.append(

            {
                "rol":"agent",
                "texto":
                "⚠️ Error consultando el orquestador."
            }

        )


        with out_debug:

            traceback.print_exc()



    finally:


        box_chat.value = render_mensajes()

        lbl_estado.value = ""

        btn_enviar.disabled = False




# =====================================================
# BOTÓN ENVIAR
# =====================================================

def enviar_mensaje(_):

    pregunta = txt_input.value.strip()


    if not pregunta:

        return



    if btn_enviar.disabled:

        return



    btn_enviar.disabled = True



    txt_input.value = ""



    historial_patito.append(

        {
            "rol":"user",
            "texto":pregunta
        }

    )


    box_chat.value = render_mensajes()



    lbl_estado.value = """

    <small>
    ⏳ El asistente está pensando...
    </small>

    """



    hilo = threading.Thread(

        target=ejecutar_orquestador,

        args=(pregunta,)

    )


    hilo.start()



btn_enviar.on_click(enviar_mensaje)



# =====================================================
# NUEVO CHAT
# =====================================================

def nuevo_chat(_):

    global config_patito


    historial_patito.clear()



    config_patito = {

        "configurable":{

            "thread_id":
            str(uuid.uuid4())

        }

    }



    box_chat.value = render_mensajes()



btn_nuevo.on_click(nuevo_chat)



# =====================================================
# DESPLIEGUE
# =====================================================

panel_chat = widgets.VBox(

    [

        widgets.HTML(

            """
            <div class="patito-card">

            <div class="patito-header">
            🤖 Asistente IA - Patito S.A.
            </div>
            """

        ),

        box_chat,

        lbl_estado,

        txt_input,

        btn_enviar,

        btn_nuevo,

        out_debug,

        widgets.HTML("</div>")

    ],

    layout=widgets.Layout(
        width="100%"
    )

)



display(
    widgets.HTML(CSS_CHAT)
)


display(panel_chat)

HTML(value='\n\n<style>\n\n.patito-card {\n    background:white;\n    border-radius:12px;\n    padding:16px;\n…